In [14]:
#resolving path issues for importing alfredgraph.csv
from pathlib import Path
import pandas as pd
import re

path = Path("../data/raw/alfredgraph.csv")

print(path.resolve())
print(path.exists())

raw = pd.read_csv(
    path,
    na_values=["", "."]
)

C:\Users\zakar\Jupyter_Projects\copper_project\data\raw\alfredgraph.csv
True


In [15]:
#separating time-series observation date from data vintage date & debugging.
raw["observation_date"] = pd.to_datetime(raw["observation_date"])

vintage_columns = [
    column 
    for column in raw.columns
    if re.fullmatch(r"PCOPPUSDM_\d{8}", column)
]

print(vintage_columns)

latest_column = max(vintage_columns)

print(raw.columns.tolist())

['PCOPPUSDM_20260605', 'PCOPPUSDM_20260713']
['observation_date', 'PCOPPUSDM_20260605', 'PCOPPUSDM_20260713']


In [16]:
#column subsetting and renaming.
copper = (
    raw[["observation_date", latest_column]]
    .rename(columns={
        "observation_date": "date",
        latest_column: "copper_price_usd_per_tonne"
    })
)

In [17]:
#type conversion, i.e., numeric coercion.
copper["copper_price_usd_per_tonne"] = pd.to_numeric(
    copper["copper_price_usd_per_tonne"],
    errors="coerce"
)

In [18]:
#date normalization
copper["date"] = (
    copper["date"]
    .dt.to_period("M")
    .dt.to_timestamp("M")
)

In [19]:
#data cleaning and ordering.
copper = (
    copper
    .dropna(subset=["copper_price_usd_per_tonne"])
    .sort_values("date")
    .drop_duplicates("date")
)

In [20]:
#metadata enrichment
copper["source"] = "IMF Primary Commodity Prices via ALFRED"
copper["frequency"] = "monthly"
copper["unit"] = "USD per metric tonne"
copper["vintage_date"] = pd.to_datetime(
    latest_column[-8:],
    format="%Y%m%d"
)

In [31]:
#importing the processed data as a .csv file.
copper.to_csv(
    "../data/processed/copper_price_monthly.csv",
    index=False
)

In [21]:
#config file stating assumptions & design decisions
import yaml
with open("../config/assumptions.yaml", "r", encoding="utf-8") as file:
    assumptions = yaml.safe_load(file)

base_currency = assumptions["project"]["base_currency"]
date_convention = assumptions["date_handling"]["standard_date"]

In [22]:
#producing validation record
validation_record = {
    "column_name": "copper_price_usd_per_tonne",
    "observations": len(copper),
    "start_date": copper["date"].min(),
    "end_date": copper["date"].max(),
    "missing_values": copper["copper_price_usd_per_tonne"].isna().sum(),
    "duplicate_dates": copper["date"].duplicated().sum(),
    "minimum": copper["copper_price_usd_per_tonne"].min(),
    "maximum": copper["copper_price_usd_per_tonne"].max(),
}

In [24]:
#checking validation result
pd.Series(validation_record)

column_name        copper_price_usd_per_tonne
observations                              414
start_date                1992-01-31 00:00:00
end_date                  2026-06-30 00:00:00
missing_values                              0
duplicate_dates                             0
minimum                           1377.376087
maximum                          13552.040909
dtype: object

In [25]:
#importing raw dataset for refinery production
import pandas as pd
from pathlib import Path

raw_path = Path(
    "../data/raw/"
    "statista_global_refinery_copper_production_kt_annual_raw.csv"
)

#the file has no formal columns, so standardizing the fields
raw = pd.read_csv(raw_path)

raw.columns = ["year", "refined_copper_production_kt"]

raw.head()

,year,refined_copper_production_kt
0,2000,14793
1,2001,15638
2,2002,15354
3,2003,15272
4,2004,15918


In [30]:
#cleaning the year and production fields
raw["year"] = (
    raw["year"]
    .astype(str)
    .str.replace("*", "", regex=False)
)

raw["year"] = pd.to_numeric(raw["year"], errors="coerce")

raw["refined_copper_production_kt"] = pd.to_numeric(
    raw["refined_copper_production_kt"],
    errors="coerce"
)

In [31]:
#creating a standardized annual date
production = raw.copy()

production["date"] = pd.to_datetime(
    production["year"].astype("Int64").astype(str) + "-12-31"
)

production = production[
    ["date", "refined_copper_production_kt"]
    ].sort_values("date")

In [32]:
#validating the dataset
validation_record = {
    "observations": len(production),
    "start_date": production["date"].min(),
    "end_date": production["date"].max(),
    "missing_dates": production["date"].isna().sum(),
    "missing_values": (
        production["refined_copper_production_kt"].isna().sum()
    ),
    "duplicate_dates": production["date"].duplicated().sum(),
    "minimum_value": (
        production["refined_copper_production_kt"].min()
    ),
    "maximum_value": (
        production["refined_copper_production_kt"].max()
    ),
}

pd.Series(validation_record)

observations                        25
start_date         2000-12-31 00:00:00
end_date           2024-12-31 00:00:00
missing_dates                        0
missing_values                       0
duplicate_dates                      0
minimum_value                    14793
maximum_value                    27486
dtype: object

In [33]:
#ensuring each year from 2000 to 2024 is present
expected_years = set(range(2000, 2025))
actual_years = set(production["date"].dt.year)

missing_years = expected_years - actual_years
extra_years = actual_years - expected_years

print("Missing Years:", missing_years)
print("Extra Years:", extra_years)

Missing Years: set()
Extra Years: set()


In [35]:
#confirming that the production column is numeric
production.dtypes

date                            datetime64[ns]
refined_copper_production_kt             int64
dtype: object

In [36]:
#saving the cleaned series
production.to_csv(
    "../data/processed/"
    "refined_copper_production_annual_clean.csv",
    index=False
)

In [40]:
#updating data dictionary
data_dictionary = pd.read_csv(
    "../data/metadata/data_dictionary.csv"
)

new_entry = {
    "column_name": "refined_copper_production_kt",
    "dataset": "annual copper fundamentals",
    "definition": "Global refined copper production",
    "unit": "thousand metric tonnes",
    "frequency": "annual",
    "data_type": "numeric",
    "date_convention": "calendar year-end",
    "transformation": "Cleaned Statista/ICSG series",
    "source_id": "SRC-002"
}

data_dictionary = pd.concat(
    [data_dictionary, pd.DataFrame([new_entry])],
    ignore_index=True
)

data_dictionary.to_csv(
    "../data/metadata/data_dictionary.csv",
    index=False
)

In [46]:
#updating source register

source_register = pd.read_csv(
    "../data/metadata/source_register.csv"
)

source_entry = {
    "source_id": "SRC-002",
    "variable_or_dataset": "refined_copper_production_kt",
    "provider": "Statista",
    "underlying_source": "International Copper Study Group",
    "series_name": "Global copper refinery production",
    "url": "https://www.statista.com/statistics/254917/total-global-copper-production-since-2006/",
    "frequency": "annual",
    "unit": "thousand metric tonnes",
    "access_date": "2026-08-03",
    "coverage": "2000-2024",
    "source_type": "secondary reproduction"
}

source_register = pd.concat(
    [source_register, pd.DataFrame([source_entry])],
    ignore_index=True
)

source_register.to_csv(
    "../data/metadata/source_register.csv",
    index=False
)

In [52]:
#using pdfplumber to locate annex page from ICSG report programmatically in order to extract production and refinery data
import pdfplumber

pdf_path = "../data/raw/icsg_factbook_2025_raw_annex.pdf"

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        text = page.extract_text() or ""

        if "WORLD COPPER PRODUCTION AND REFINED COPPER USAGE" in text:
            print("Found annex on page:", page_number)
            print(text[:3000])

In [54]:
#inspecting raw extracted table
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[page_number]

    table = page.extract_table(
        table_settings={
            "vertical_strategy": "text",
            "horizontal_strategy": "text"
        }
    )

for row in table:
    print(row)

TypeError: 'NoneType' object is not iterable

In [55]:
#pdfplumber likely did not detect targetted table, undertaking a diagnostic check
if table is None:
    print("No table detected on this page.")
else:
    for row in table:
        print(row)

No table detected on this page.


In [57]:
#inspecting the page as text
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[page_number]

    text = page.extract_text(layout=True)

print(text)

In [58]:
#page_number is likely pointing to the last PDF page. Assuming the cause is that page_number was used as the loop variable whilst searching, i.e., no page match or after finding the match, the loop continued.
#attempting to use a separate variable and stop once the annex is found
import pdfplumber

annex_page_number = None

with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        normalized_text = " ".join(text.split()).lower()

        if (
            "world copper production and refined copper usage"
            in normalized_text
        ):
            annex_page_number = page_index
            break

if annex_page_number is None:
    raise ValueError("Annex page was not found")

print("Python page index:", annex_page_number)
print("PDF page number:", annex_page_number + 1)

Python page index: 1
PDF page number: 2


In [59]:
with pdfplumber.open(pdf_path) as pdf:
    annex_page = pdf.pages[annex_page_number]
    print(annex_page.extract_text(layout=True))

                                                                                                                    
                                                                                                                    
                                                                                                                    
                                             The World Copper Factbook 2025                                         
                                                                                                                    
                                                                                                                    
          Table of Contents                                  Chapter 5: Copper Trade ........................................................................................ 28
                                                              International Trade Flow of Copper Ores and Concentrates, 2

In [60]:
#that didn't work lol, let me use more distinctive text from the annex
annex_candidates = []

with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        normalized_text = " ".join(text.split()).lower()

        if (
            "world copper production and refined copper usage"
            in normalized_text
            and "thousand metric tonnes copper"
            in normalized_text
            and "mine production" in normalized_text
            and "refined usage" in normalized_text
        ):
            annex_candidates.append(page_index)

print("Candidate pages:", annex_candidates)
print(
    "PDF page numbers:",
    [page_index + 1 for page_index in annex_candidates]
)

Candidate pages: []
PDF page numbers: []


In [61]:
#no candidate pages, inspecting all pages containing the broad phrase
with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        normalized_text = " ".join(text.split()).lower()

        if "world copper production" in normalized_text:
            print(
                "Python index:",
                page_index,
                "PDF page:",
                page_index + 1
            )

Python index: 1 PDF page: 2


In [62]:
#the broad search finds only the table of contents; the stricter search finds nothing. Thus, the annex page must be either on a different page than expected or its table content is not exposed to pdfplumber as ordinary text.
#first, let's inspect the last several pages directly. The annex should be near the end of the 68 pg PDF.
import pdfplumber

with pdfplumber.open(pdf_path) as pdf:
    print("Total pages:", len(pdf.pages))

    for page_index in range(max(0, len(pdf.pages) - 10), len(pdf.pages)):
        text = pdf.pages[page_index].extract_text() or ""

        print("\n--- Python index:", page_index,
              "| PDF page:", page_index + 1, "---")
        print(text[:500])

Total pages: 68

--- Python index: 58 | PDF page: 59 ---
The World Copper Factbook 2025
C R R D “metallurgical” indicator are the metal industry, metal traders, and
OPPER ECYCLING ATE EFINITIONS
resource policymakers. However, given structural and process
variables, it may have limited use as a policy tool.
•• The Overall Recycling Efficiency Rate indicates the efficiency with
which end-of-life (EoL) scrap, new scrap, and other metal-bearing
residues are collected and recycled by a network of collectors,
processors, and metal recyclers. The key target

--- Python index: 59 | PDF page: 60 ---
The World Copper Factbook 2025
ICSG G C S U R I R , 2005-2024
LOBAL OPPER CRAP SAGE AND ECYCLING NPUT ATE
Million metric tonnes of copper
Source: ICSG
Direct-melt copper scrap figures for 2024 are preliminary estimates, as is the Recycling Input Rate (RIR).
In ternatio nal Cop per Stu dy Group 55
(
3
3
2
2
2
1
1
M
6
2
8
4
0
6
2
8
4
0
t C
5002
u )
6002 7002
D
( S e
8002
ir e
c
c
o
t
n
9002
M
d a
e
r

In [63]:
#found it on python index: 64, and pdf page: 65. However, data is poorly extracted. Distorted headings are likely a layout extraction issue, as opposed to a data issue. Parsing now:
with pdfplumber.open(pdf_path) as pdf:
    annex_page = pdf.pages[64]
    annex_text = annex_page.extract_text(layout=False)

print(annex_text)

The World Copper Factbook 2025
ANNEX
W C P R C U , 1960-2024
ORLD OPPER RODUCTION AND EFINED OPPER SAGE
Thousand Metric Tonnes Copper
Source: ICSG
Mine Refined Refined Mine R efined Refined Mine Refined Refined
Production Production Usage Production Production Usage Production Production Usage
1960 3,924 4,998 4,738 1982 7,745 9,319 9,090 2004 14,594 15,918 16,743
1961 4,081 5,127 5,050 1983 7,824 9,541 9,510 2005 14,927 16,572 16,552
1962 4,216 5,296 5,048 1984 8,135 9,440 9,930 2006 14,983 17,288 16,917
1963 4,286 5,400 5,500 1985 8,314 9,616 9,798 2007 15,508 17,895 18,026
1964 4,443 5,739 5,995 1986 8,295 9,920 10,112 2008 15,532 18,191 17,877
1965 4,769 6,059 6,193 1987 8,620 10,148 10,293 2009 15,941 18,234 17,870
1966 4,987 6,324 6,445 1988 8,773 10,512 10,668 2010 15,987 18,965 19,136
1967 4,743 6,004 6,195 1989 9,086 10,908 11,081 2011 15,960 19,585 19,709
1968 5,010 6,653 6,523 1990 9,227 10,805 10,886 2012 16,678 20,169 20,479
1969 5,682 7,212 7,137 1991 9,373 10,686 10,563 

In [65]:
#numerical rows look properly aligned. Proceeding with extracting the data rows.
import re
import pandas as pd

records = []

for line in annex_text.splitlines():
    tokens = line.split()

    #data rows begin with a four-digit year
    if not tokens or not re.match(r"^\d{4}", tokens[0]):
        continue

    #each block contains year + three values
    for position in range(0, len(tokens), 4):
        block = tokens[position:position + 4]

        if len(block) !=4:
            continue

        year_token, mine, refined, usage = block

        year = int(re.sub(r"\D", "", year_token))

        records.append({
            "year": year,
            "copper_mine_production_kt": mine.replace(",", ""),
            "refined_copper_production_kt": refined.replace(",", ""),
            "refined_copper_usage_kt": usage.replace(",", ""),
            "preliminary": "/p" in year_token.lower()
        })

In [66]:
#converting the values to numeric types
annual = pd.DataFrame(records)

numeric_columns = [
    "copper_mine_production_kt",
    "refined_copper_production_kt",
    "refined_copper_usage_kt"
]

for column in numeric_columns:
    annual[column] = pd.to_numeric(
        annual[column],
        errors="coerce"
    )

annual["date"] = pd.to_datetime(
    annual["year"].astype(str) + "-12-31"
)

annual = annual[
    [
        "date",
        "year",
        "copper_mine_production_kt",
        "refined_copper_production_kt",
        "refined_copper_usage_kt",
        "preliminary"
    ]
].sort_values("year")

In [67]:
#validating
print(annual.head())
print(annual.tail())
print(annual.shape)
print(annual["year"].is_unique)
print(annual["year"].min(), annual["year"].max())

         date  year  copper_mine_production_kt  refined_copper_production_kt  \
0  1960-12-31  1960                       3924                          4998   
3  1961-12-31  1961                       4081                          5127   
6  1962-12-31  1962                       4216                          5296   
9  1963-12-31  1963                       4286                          5400   
12 1964-12-31  1964                       4443                          5739   

    refined_copper_usage_kt  preliminary  
0                      4738        False  
3                      5050        False  
6                      5048        False  
9                      5500        False  
12                     5995        False  
         date  year  copper_mine_production_kt  refined_copper_production_kt  \
50 2020-12-31  2020                      20740                         24621   
53 2021-12-31  2021                      21223                         24900   
56 2022-12-31  2022  

In [68]:
#index retains the original extraction order, reseting it
annual = (
    annual
    .sort_values("year")
    .reset_index(drop=True)
)

In [69]:
#inspecting key recent years
recent = annual[annual["year"] >= 2019].copy()

recent

,date,year,copper_mine_production_kt,refined_copper_production_kt,refined_copper_usage_kt,preliminary
59,2019-12-31,2019,20657,24127,24351,False
60,2020-12-31,2020,20740,24621,24953,False
61,2021-12-31,2021,21223,24900,25259,False
62,2022-12-31,2022,21911,25272,25857,False
63,2023-12-31,2023,22368,26502,26604,False
64,2024-12-31,2024,22990,27486,27353,True


In [70]:
#calculating refined copper balance
recent["refined_copper_balance_kt"] = (
    recent["refined_copper_production_kt"]
    - recent["refined_copper_usage_kt"]
)

recent[
    [
        "year",
        "refined_copper_production_kt",
        "refined_copper_usage_kt",
        "refined_copper_balance_kt",
        "preliminary"
    ]
]

,year,refined_copper_production_kt,refined_copper_usage_kt,refined_copper_balance_kt,preliminary
59,2019,24127,24351,-224,False
60,2020,24621,24953,-332,False
61,2021,24900,25259,-359,False
62,2022,25272,25857,-585,False
63,2023,26502,26604,-102,False
64,2024,27486,27353,133,True


In [71]:
annual.to_csv(
    "../data/processed/"
    "icsg_world_copper_production_usage_annual_clean.csv",
    index=False
)

In [73]:
#validating against raw statisa dataset, treating it as an independent validation source
#loading the two datasets
import pandas as pd
import numpy as np

icsg = pd.read_csv(
    "../data/processed/"
    "icsg_world_copper_production_usage_annual_clean.csv"
)

statista = pd.read_csv(
    "../data/raw/"
    "statista_global_refinery_copper_production_kt_annual_raw.csv"
)

In [74]:
#standardizing statista file
statista.columns = [
    "year",
    "refined_copper_production_kt_statista"
]

statista["year"] = (
    statista["year"]
    .astype(str)
    .str.replace("*", "", regex=False)
    .astype(int)
)

statista["refined_copper_production_kt_statista"] = (
    pd.to_numeric(
        statista["refined_copper_production_kt_statista"],
        errors="coerce"
    )
)

In [75]:
#merging relevant ICSG column
comparison = icsg[
    [
        "year",
        "refined_copper_production_kt"
    ]
].merge(
    statista,
    on="year",
    how="outer",
    indicator=True
)

In [76]:
#calculating differences
comparison["difference_kt"] = (
    comparison["refined_copper_production_kt"]
    - comparison["refined_copper_production_kt_statista"]
)

comparison["percentage_difference"] = (
    comparison["difference_kt"]
    / comparison["refined_copper_production_kt_statista"]
    * 100
)

In [77]:
comparison

,year,refined_copper_production_kt,refined_copper_production_kt_statista,_merge,difference_kt,percentage_difference
0,1960,4998,NaN,left_only,NaN,NaN
1,1961,5127,NaN,left_only,NaN,NaN
2,1962,5296,NaN,left_only,NaN,NaN
3,1963,5400,NaN,left_only,NaN,NaN
4,1964,5739,NaN,left_only,NaN,NaN
...,...,...,...,...,...,...
60,2020,24621,24621.0,both,0.0,0.0
61,2021,24900,24900.0,both,0.0,0.0
62,2022,25272,25272.0,both,0.0,0.0
63,2023,26502,26502.0,both,0.0,0.0


In [78]:
#no differences upon manual inspection
#checking whether all overlapping values match exactly
overlap = comparison[
    comparison["_merge"] == "both"
]

print("Maximum absolute difference:",
      overlap["difference_kt"].abs().max())

print("Number of differences:",
      (overlap["difference_kt"] != 0).sum())

Maximum absolute difference: 0.0
Number of differences: 0


In [81]:
#two files match exactly per output above.
#formally verifying it:
assert overlap["difference_kt"].eq(0).all()

In [82]:
#checking non overlapping years:
comparison[
    comparison["_merge"] != "both"
][["year", "_merge"]]

,year,_merge
0,1960,left_only
1,1961,left_only
2,1962,left_only
3,1963,left_only
4,1964,left_only
5,1965,left_only
6,1966,left_only
7,1967,left_only
8,1968,left_only
9,1969,left_only


In [83]:
#saving the comparison as a validation artifact
comparison.to_csv(
    "../data/processed/"
    "icsg_vs_statista_production_validation.csv",
    index=False
)

In [88]:
#running final checks
assert annual["year"].between(1960, 2024).all()
assert annual["year"].is_unique
assert annual[numeric_columns].notna().all().all()
assert annual["year"].min() == 1960
assert annual["year"].max() == 2024

#calculating derived refined balance
annual["refined_copper_balance_kt"] = (
    annual["refined_copper_production_kt"]
    - annual["refined_copper_usage_kt"]
)

annual_analysis = annual.copy()

annual_analysis["refined_copper_balance_kt"] = (
    annual_analysis["refined_copper_production_kt"]
    - annual_analysis["refined_copper_usage_kt"]
)

#exporting derived analytical output as a .csv file
annual_analysis.to_csv(
    "../data/processed/"
    "copper_fundamentals_annual_analysis.csv",
    index=False
)

In [89]:
#saving a final quality-control summary:
validation_summary = {
    "observations": len(annual),
    "start_year": annual["year"].min(),
    "end_year": annual["year"].max(),
    "duplicate_years": annual["year"].duplicated().sum(),
    "missing_values": annual.isna().sum().sum(),
    "statista_production_discrepancies": (
        comparison["difference_kt"] != 0
    ).sum()
}

pd.Series(validation_summary).to_csv(
    "../data/processed/"
    "copper_fundamentals_validation_summary.csv"
)

In [92]:
# Used Copper Production: World Refined Copper Production Mountain graph, p. 22, from ICSG's World Copper Factbook 2025, in conjunction with WebPlotDigitizer, to draw two boundaries. First one drawn at values of Refinery Primary; second at values of Refinery Secondary. Thus, I will take the difference between the secondary and primary, in order to approximate values for secondary/recycled copper production; then validate using publicly available data.
# Importing corresponding .csv files for both primary and secondary values. The files do not contain column headers; therefore providing column names myself.
import pandas as pd
import numpy as np

primary = pd.read_csv(
    "../data/raw/primary_cumulative_boundary_mt.csv",
    header=None,
    names=["year_raw", "primary_boundary_mt"]
)

secondary_top = pd.read_csv(
    "../data/raw/primary_plus_secondary_boundary_mt.csv",
    header=None,
    names=[
        "year_raw",
        "primary_plus_secondary_boundary_mt"
    ]
)

In [93]:
# Inspecting assigned column names.
print(primary.head())
print(secondary_top.head())
print(primary.shape)
print(secondary_top.shape)

      year_raw  primary_boundary_mt
0  1960.091822             3.733333
1  1961.010043             3.845161
2  1962.020086             3.938351
3  1963.011765             4.087455
4  1964.003443             4.348387
      year_raw  primary_plus_secondary_boundary_mt
0  1960.036729                            4.422939
1  1961.028407                            4.572043
2  1962.020086                            4.646595
3  1963.030129                            4.944803
4  1964.040172                            5.205735
(65, 2)
(65, 2)


In [94]:
# Output looks about right.
# Converting fields.
import numpy as np

for df in [primary, secondary_top]:
    df["year_raw"] = pd.to_numeric(
        df["year_raw"],
        errors="coerce"
    )

primary["primary_boundary_mt"] = pd.to_numeric(
    primary["primary_boundary_mt"],
    errors="coerce"
)

secondary_top[
    "primary_plus_secondary_boundary_mt"
] = pd.to_numeric(
    secondary_top[
        "primary_plus_secondary_boundary_mt"
    ],
    errors="coerce"
)

In [95]:
# Creating integer years.
primary["year"] = np.rint(
    primary["year_raw"]
).astype(int)

secondary_top["year"] = np.rint(
    secondary_top["year_raw"]
).astype(int)

In [96]:
# Retaining the fields needed for merging
primary_for_merge = primary[
    ["year", "primary_boundary_mt"]
].copy()

secondary_for_merge = secondary_top[
    [
        "year",
        "primary_plus_secondary_boundary_mt"
    ]
].copy()

In [97]:
# Merging the two extracted boundaries.

digitized = primary_for_merge.merge(
    secondary_for_merge,
    on="year",
    how="outer",
    indicator=True,
    validate="one_to_one"
)

In [98]:
# Ensuring that each year appears in both files.

print(digitized["_merge"].value_counts())

if not digitized["_merge"].eq("both").all():
    print(
        digitized[
            digitized["_merge"] != "both"
        ]
    )
    raise ValueError(
        "Some years do not appear in both extracted datasets."
    )

_merge
both          65
left_only      0
right_only     0
Name: count, dtype: int64


In [99]:
# Approximating secondary refined production

digitized["secondary_refined_production_mt"] = (
    digitized[
        "primary_plus_secondary_boundary_mt"
    ]
    - digitized["primary_boundary_mt"]
)

In [100]:
# Converting from million tonnes to thousand tonnes.

digitized["secondary_refined_production_kt"] = (
    digitized["secondary_refined_production_mt"] * 1000
)

In [101]:
# Adding standardized dates and sorting.

digitized["date"] = pd.to_datetime(
    digitized["year"].astype(str) + "-12-31"
)

digitized = (
    digitized
    .sort_values("year")
    .reset_index(drop=True)
)

In [102]:
# Quick quality checks.

assert len(digitized) == 65
assert digitized["year"].min() == 1960
assert digitized["year"].max() == 2024
assert digitized["year"].is_unique
assert digitized[
    "secondary_refined_production_mt"
].ge(0).all()

In [103]:
# Inspecting results.

digitized[
    [
        "year",
        "primary_boundary_mt",
        "primary_plus_secondary_boundary_mt",
        "secondary_refined_production_mt",
        "secondary_refined_production_kt"
    ]
].head()

,year,primary_boundary_mt,primary_plus_secondary_boundary_mt,secondary_refined_production_mt,secondary_refined_production_kt
0,1960,3.733333,4.422939,0.689606,689.605735
1,1961,3.845161,4.572043,0.726882,726.881720
2,1962,3.938351,4.646595,0.708244,708.243728
3,1963,4.087455,4.944803,0.857348,857.347670
4,1964,4.348387,5.205735,0.857348,857.347670


In [104]:
# Inspecting recent years.

digitized[
    digitized["year"] >= 2020
]

,year,primary_boundary_mt,primary_plus_secondary_boundary_mt,_merge,secondary_refined_production_mt,secondary_refined_production_kt,date
60,2020,16.649462,20.488889,both,3.839427,3839.426523,2020-12-31
61,2021,16.761290,20.898925,both,4.137634,4137.634409,2021-12-31
62,2022,16.873118,20.973477,both,4.100358,4100.358423,2022-12-31
63,2023,17.469534,21.961290,both,4.491756,4491.756272,2023-12-31
64,2024,18.065950,22.762724,both,4.696774,4696.774194,2024-12-31


In [105]:
# Outputs seem structurally correct and economically plausible. 2024 figure is especially reassuring, relative to total refined production, i.e. secondary share is approximately 17.1%, which matches the figure quoted in ICSG's Factbook.
# Running additional checks.

assert digitized["_merge"].eq("both").all()
assert digitized["secondary_refined_production_mt"].ge(0).all()
assert digitized["year"].is_unique
assert digitized["year"].min() == 1960
assert digitized["year"].max() == 2024

In [106]:
# Since the values were digitized from a chart, I won't present them with excessive precision. Rather, I'll preserve the raw extracted values, but create a rounded reporting figure in application.
digitized[
    "secondary_refined_production_mt_rounded"
] = digitized[
    "secondary_refined_production_mt"
].round(2)

digitized[
    "secondary_refined_production_kt_rounded"
] = (
    digitized[
        "secondary_refined_production_kt"
    ].round(-1)
)

In [107]:
# Removing merge-status column from final output.
secondary_clean = digitized[
    [
        "date",
        "year",
        "secondary_refined_production_mt",
        "secondary_refined_production_kt",
        "secondary_refined_production_mt_rounded",
        "secondary_refined_production_kt_rounded"
    ]
].copy()

In [108]:
# Saving as a digitized derived series.
secondary_clean.to_csv(
    "../data/processed/"
    "secondary_refined_production_annual_digitized.csv",
    index=False
)